# 01 · Build the analysis table

Input files are private (Dimensions terms of use and personal data). See `data/README.md`.

| File | Content |
|---|---|
| `data/raw/grants_screened.xlsx` | Dimensions export of 13 Feb 2024 after screening and hand enrichment (818 grants, 1978–2024) |
| `data/raw/receiver_metadata.xlsx` | One row per recipient institution: country, region, type, World Bank income group, Global North/South |
| `data/raw/gender_lookup.xlsx` | One row per researcher name and its inferred gender (Gender API plus manual checks) |
| `data/interim/topic_assignments.csv` | Grant ID and LDA topic, from `02_topic_modelling.ipynb` |

**Output:** `data/processed/phage_funding_analysis.csv`, holding 802 grants that started 1999–2023.

**Change from the 2024–25 notebook:** lists now keep their order. The new `receiver_*_list` columns hold one value per institution, in the same order as `receiver_org_list`, so funding can be split fairly between institutions. The legacy columns are still built, so results can be compared with earlier versions.

Run `02_topic_modelling.ipynb` first. It writes the topic file this notebook needs.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
RAW, INTERIM, PROCESSED = (ROOT / "data" / p for p in ("raw", "interim", "processed"))
TAB, FIG = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures"
for p in (INTERIM, PROCESSED, TAB, FIG):
    p.mkdir(parents=True, exist_ok=True)
import preprocess as pp

In [2]:
d = pp.build(RAW / "grants_screened.xlsx", RAW / "receiver_metadata.xlsx",
             RAW / "gender_lookup.xlsx", INTERIM / "topic_assignments.csv")
print(f"{len(d)} grants, {d['Start Year'].min()}–{d['Start Year'].max()}, USD {d['Funding amount in USD'].sum()/1e6:.1f}M")

802 grants, 1999–2023, USD 318.6M


## Checks
Only totals are shown here. The table holds personal names, so no grant-level rows are printed.

In [3]:
checks = pd.Series({
    "grants": len(d),
    "unique Grant IDs": d["Grant ID"].nunique(),
    "missing USD amount": int(d["Funding amount in USD"].isna().sum()),
    "grants with >1 recipient institution": int((d["n_recipients"] > 1).sum()),
    "institutions without metadata": int(d["receiver_income_list"].str.split(";").explode().eq("").sum()),
    "grants with no named researcher": int((d["n_researchers"] == 0).sum()),
    "researcher names without gender": int(d["n_gender_unknown"].sum()),
    "grants without topic": int(d["topic_id"].isna().sum()),
    "topics inherited from a duplicate text": int(d["topic_source"].astype(str).str.startswith("duplicate").sum()),
})
assert checks["grants"] == checks["unique Grant IDs"] == 802
checks

grants                                    802
unique Grant IDs                          802
missing USD amount                          0
grants with >1 recipient institution      135
institutions without metadata               0
grants with no named researcher            59
researcher names without gender             3
grants without topic                        0
topics inherited from a duplicate text     17
dtype: int64

In [4]:
pd.DataFrame({
    "Team gender": d["Gender"].fillna("No named researcher").value_counts(),
}).join(d["Topic N°"].value_counts().rename("Topic"), how="outer")

,Team gender,Topic
Gender,,
Antibiotic Resistance and Therapeutic Strategies,NaN,151.0
Bacterial Genetics and Mechanisms,NaN,237.0
Bacteriophage Therapy and Infectious Disease Control,NaN,280.0
Female,201.0,NaN
Female;Male,95.0,NaN
Immunotherapy and Targeted Treatments,NaN,133.0
Male,446.0,NaN
No named researcher,60.0,NaN
Not Class.,NaN,1.0


## Comparison with the April 2025 table (optional)
This runs only if the old file `data/processed/phage_funding_preprocesse.csv` is present.

In [5]:
legacy = PROCESSED / "phage_funding_preprocesse.csv"
if legacy.exists():
    o = pd.read_csv(legacy).set_index("Grant ID"); n = d.set_index("Grant ID").loc[o.index]
    as_set = lambda s: s.fillna("").astype(str).apply(lambda x: frozenset(v.strip() for v in x.split(";") if v.strip() not in ("", "nan")))
    cols = ["Gender", "Receiver - Country of standardized research organization", "Receiver - Income group",
            "Receiver - Global north - south", "Topic N°"]
    print(pd.Series({c: f"{(as_set(o[c]) == as_set(n[c])).mean()*100:.1f}% identical" for c in cols}).to_string())
    print("USD identical:", bool((o["Funding amount in USD"] == n["Funding amount in USD"]).all()))

Gender                                                       99.9% identical
Receiver - Country of standardized research organization    100.0% identical
Receiver - Income group                                     100.0% identical
Receiver - Global north - south                             100.0% identical
Topic N°                                                     97.9% identical
USD identical: True


In [6]:
d.to_csv(PROCESSED / "phage_funding_analysis.csv", index=False)
print("written data/processed/phage_funding_analysis.csv")

written data/processed/phage_funding_analysis.csv
